In [ ]:
!pip install chromadb sentence_transformers groq -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not current

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import chromadb
import os
from groq import Groq



GROQ_API_KEY="YOUR_GROQ_API_KEY"

os.environ['GROQ_API_KEY'] = GROQ_API_KEY
groq_client = Groq(api_key=GROQ_API_KEY)


In [ ]:
df=pd.read_csv('college_notes.csv')
print(df.shape, df.columns.tolist(),df.head(3))

(15, 4) ['note_id', 'subject', 'topic', 'content']   note_id  ...                                            content
0    N001  ...  ETL stands for Extract Transform Load. It is t...
1    N002  ...  A database is an organized collection of data ...
2    N003  ...  Data cleaning involves fixing or removing inco...

[3 rows x 4 columns]


In [ ]:
print(df['subjects'].value_counts())

In [ ]:
documents = df['content'].tolist()
metadatas = df[['subject', 'topic']].to_dict(orient='records')
ids = df['note_id'].astype(str).tolist()

print(f"Total chunks: {len(documents)}")
print(f"First doc ID: {ids[0]}")

Total chunks: 15
First doc ID: N001


In [ ]:
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="college_notes")

collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print(f"Collection count: {collection.count()}")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 97.9MiB/s]


Collection count: 15


In [ ]:
from chromadb.utils import embedding_functions

# Define the embedding function
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

# Create/Reset the collection with the embedding function
# We use a new name or delete the old one to ensure the schema is updated
collection_with_emb = chroma_client.get_or_create_collection(
    name="college_notes_embedded",
    embedding_function=sentence_transformer_ef
)

# Add documents (embeddings will be generated automatically by the EF)
collection_with_emb.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print(f"New collection count: {collection_with_emb.count()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


New collection count: 15


In [ ]:
# Update search function to use the new collection
def search_notes_v2(query, n_results=2):
    results = collection_with_emb.query(
        query_texts=[query],
        n_results=n_results
    )
    return results

# Test retrieval with embeddings
test_query = "What is ETL?"
res = search_notes_v2(test_query)
print(f"Retrieved: {res['documents'][0][0]}")

Retrieved: ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehouse for analysis.


In [ ]:
def search_notes(query, n_results=2):
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )
    return results

# Test search
query = "What is ETL?"
search_results = search_notes(query)
print(f"Search results for: {query}")
for doc in search_results['documents'][0]:
    print(f"- {doc}")

Search results for: What is ETL?
- ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehouse for analysis.
- An API or Application Programming Interface allows two software applications to talk to each other. In data engineering APIs are used to fetch data from external services like weather data stock prices or social media feeds.


In [ ]:
def generate_answer_v2(query):
    # Use the new collection with embeddings
    results = search_notes_v2(query, n_results=3)
    context = "\n".join(results['documents'][0])

    prompt = f"""You are a helpful college assistant. Use the following context to answer the student's question.\n\nContext:\n{context}\n\nQuestion: {query}"""

    completion = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.5
    )
    return completion.choices[0].message.content

# Final test with embedding-based retrieval
question = "Explain ETL and how it relates to Data Engineering."
answer = generate_answer_v2(question)
print(f"Question: {question}\n")
print(f"AI Answer (Using Embeddings):\n{answer}")

Question: Explain ETL and how it relates to Data Engineering.

AI Answer (Using Embeddings):
I'd be happy to explain ETL and its connection to Data Engineering.

**What is ETL?**

ETL stands for Extract, Transform, Load. It's a process used to collect raw data from different sources, transform it into a clean and structured format, and load it into a database or data warehouse for analysis. The ETL process involves three main steps:

1. **Extract**: This step involves collecting raw data from various sources, such as databases, files, or external APIs. The goal is to gather all the necessary data required for analysis.
2. **Transform**: In this step, the extracted data is cleaned, processed, and transformed into a standardized format. This may involve data normalization, aggregation, or filtering.
3. **Load**: The final step involves loading the transformed data into a database or data warehouse, where it can be analyzed and queried.

**How does ETL relate to Data Engineering?**

Data 

In [ ]:
emb=SentenceTransformer("all-MiniLM-L6-v2")
print(emb.encode("hello").shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(384,)


In [ ]:
def retrieve_relevant_chunks(test_question, top_k=3):
    results = collection_with_emb.query(
        query_texts=[test_question],
        n_results=top_k
    )
    return results

# Run the function with a test question
test_question = "What is RAG?"
retrieval_results = retrieve_relevant_chunks(test_question, top_k=3)

print(f"Results for: {test_question}")
for i, doc in enumerate(retrieval_results['documents'][0]):
    print(f"Chunk {i+1}: {doc}")

Results for: What is RAG?
Chunk 1: RAG or Retrieval Augmented Generation is a technique where an AI model first retrieves relevant documents from a knowledge base and then generates an answer based on those retrieved documents. This reduces hallucination and allows AI to answer questions about specific data.
Chunk 2: Data cleaning involves fixing or removing incorrect incomplete duplicate or corrupted data. Common cleaning tasks include handling missing values removing duplicates fixing data types and standardizing formats.
Chunk 3: ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehouse for analysis.


In [ ]:
for i in range(len(retrieval_results['documents'][0])):
    doc = retrieval_results['documents'][0][i]
    distance = retrieval_results['distances'][0][i]
    metadata = retrieval_results['metadatas'][0][i]

    print(f"--- Result {i+1} ---")
    print(f"Document: {doc}")
    print(f"Distance: {distance}")
    print(f"Metadata: {metadata}")
    print(f"Subject: {metadata.get('subject')}")
    print(f"Topic: {metadata.get('topic')}")
    print(f"Content: {doc}")
    print("\n")

--- Result 1 ---
Document: RAG or Retrieval Augmented Generation is a technique where an AI model first retrieves relevant documents from a knowledge base and then generates an answer based on those retrieved documents. This reduces hallucination and allows AI to answer questions about specific data.
Distance: 0.5026322603225708
Metadata: {'topic': 'Retrieval Augmented Generation', 'subject': 'Generative AI'}
Subject: Generative AI
Topic: Retrieval Augmented Generation
Content: RAG or Retrieval Augmented Generation is a technique where an AI model first retrieves relevant documents from a knowledge base and then generates an answer based on those retrieved documents. This reduces hallucination and allows AI to answer questions about specific data.


--- Result 2 ---
Document: Data cleaning involves fixing or removing incorrect incomplete duplicate or corrupted data. Common cleaning tasks include handling missing values removing duplicates fixing data types and standardizing formats.
Di

In [ ]:
def build_context(results):
    """Combines retrieved document chunks into a single context string."""
    context_parts = []
    for doc in results['documents'][0]:
        context_parts.append(doc)
    return "\n\n".join(context_parts)

# Run the function
context_string = build_context(retrieval_results)
print("--- Generated Context ---")
print(context_string)

--- Generated Context ---
RAG or Retrieval Augmented Generation is a technique where an AI model first retrieves relevant documents from a knowledge base and then generates an answer based on those retrieved documents. This reduces hallucination and allows AI to answer questions about specific data.

Data cleaning involves fixing or removing incorrect incomplete duplicate or corrupted data. Common cleaning tasks include handling missing values removing duplicates fixing data types and standardizing formats.

ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehouse for analysis.


In [ ]:
def generate_rag_answer(question,context):
  """
  send the retrieved context and question to groq llm for answer generation

  parameters:
  question(str): the user question
  context(str): the retrieved chunks

  Returns:
      answer(str): The LLM's generated answer
  """

  system_prompt = """
You are a helpful academic assistant for engineering students.

Your role is to answer student questions using only the provided context.

Guidelines:
1. Use the retrieved context to generate accurate answers.
2. If the answer is not available in the context, say:
   "I could not find relevant information in the notes."
3. Keep answers clear, concise, and beginner-friendly.
4. Explain technical concepts in simple language.
5. Use proper academic tone.
6. Do not generate unrelated or hallucinated information.
7. If multiple topics are retrieved, combine them logically.
8. Focus on engineering subjects such as:
   - Data Engineering
   - Machine Learning
   - Cloud Computing
   - Generative AI

You will receive:
- Retrieved context from semantic search
- A user question

Generate the best possible answer based on the context.
"""

  user_prompt = f"""
Context:
{context}

Question:
{question}

Answer the question using only the provided context.
Explain the answer clearly for an engineering student.
"""

  response = groq_client.chat.completions.create(
      model = "llama-3.1-8b-instant",
      messages = [
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": user_prompt}
      ],
      temperature = 0.1,
      max_tokens = 500
  )

  answer = response.choices[0].message.content
  return answer

print("RAG generation function defined")


RAG generation function defined


In [ ]:
final_answer = generate_rag_answer(test_question, context_string)

print(f"Question: {test_question}\n")
print("--- Final RAG Answer ---")
print(final_answer)

Question: What is RAG?

--- Final RAG Answer ---
Based on the provided context, RAG stands for Retrieval Augmented Generation. It's a technique used by AI models to improve their accuracy when answering questions about specific data.

Here's how it works: the AI model first retrieves relevant documents from a knowledge base. Think of a knowledge base as a massive library of information. The AI model searches through this library to find the most relevant documents related to the question being asked.

Once the AI model has retrieved these relevant documents, it then uses this information to generate an answer. This approach reduces the likelihood of the AI model "hallucinating" or making things up, as it's relying on actual information from the knowledge base.

In simpler terms, RAG is a way for AI models to be more accurate and reliable when answering questions by leveraging existing knowledge and information.


In [ ]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
import os

# Securely fetch the API key
api_key = globals().get('GROQ_API_KEY', "YOUR_GROQ_API_KEY")

print("Loading Dataset....")
df = pd.read_csv("college_notes.csv")

documents = df['content'].tolist()
ids = [f"note_{row['note_id']}" for row in df.to_dict('records')]
metadatas = [{"subject": row['subject'], "topic": row['topic']} for row in df.to_dict('records')]

print("Loading Embedding Model....")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print("Generating Embeddings....")
embeddings = embedding_model.encode(documents, show_progress_bar=False)

print("Indexing into ChromaDB....")
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="college_notes_rag_final")

collection.add(
    documents=documents,
    embeddings=embeddings.tolist(),
    ids=ids,
    metadatas=metadatas
)

def retrieve_relevant_chunks(question, top_k=3):
    question_embedding = embedding_model.encode(question).tolist()
    return collection.query(query_embeddings=[question_embedding], n_results=top_k)

def build_context_from_results(results):
    context_parts = []
    for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
        context_parts.append(f"Source {i+1} (Topic: {meta['topic']}): {doc}")
    return "\n\n".join(context_parts)

client = Groq(api_key=api_key)

def ask_college_assistant(question):
    results = retrieve_relevant_chunks(question)
    context = build_context_from_results(results)

    system_prompt = "You are a helpful academic assistant. Answer the question using ONLY the provided context."
    user_prompt = f"Context:\n{context}\n\nQuestion:\n{question}"

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    return response.choices[0].message.content

# Target Question
question = "What is ETL and why is it important in data engineering?"
answer = ask_college_assistant(question)

print(f"\nQuestion: {question}")
print(f"\nAnswer:\n{answer}")

Loading Dataset....
Loading Embedding Model....


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generating Embeddings....
Indexing into ChromaDB....

Question: What is ETL and why is it important in data engineering?

Answer:
According to Source 1, ETL (Extract, Transform, Load) is the process of collecting raw data from different sources, transforming it into a clean and structured format, and loading it into a database or data warehouse for analysis.

ETL is important in data engineering because it enables the collection of data from various sources and converts it into a consistent, usable format. This process is crucial for analyzing data, as raw data can be unstructured, incomplete, or inconsistent. The transformation step helps to resolve data inconsistencies and makes the data more meaningful for analysis.

In data engineering, ETL plays a vital role in preparing data for visualization and further analysis. By applying this ETL process, data engineers can ensure that the data is reliable, consistent, and in a format that can be easily used for data visualization and other 